<h2 align='center' style='color:purple'>Finding best model and hyper parameter tunning using GridSearchCV</h2>

**For iris flower dataset in sklearn library, we are going to find out best model and best hyper parameters using GridSearchCV**

<img src='iris_petal_sepal.png' height=300 width=300 />

**Load iris flower dataset**

In [17]:
from sklearn import datasets

iris=datasets.load_iris()

In [18]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [19]:
import pandas as pd
df=pd.DataFrame(iris.data,columns=iris.feature_names)
df["flower"]=iris.target
df["flower"]=df["flower"].apply(lambda x: iris.target_names[x])
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [20]:
df[47:52]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
47,4.6,3.2,1.4,0.2,setosa
48,5.3,3.7,1.5,0.2,setosa
49,5.0,3.3,1.4,0.2,setosa
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor


<h3 style='color:blue'>Approach 1: Use train_test_split and manually tune parameters by trial and error</h3>

In [21]:
X=iris.data
y=iris.target

In [22]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [23]:
from sklearn import svm 
model=svm.SVC(kernel='rbf',C=30,gamma='auto')
model.fit(X_train,y_train)
model.score(X_test,y_test)


0.9333333333333333

<h3 style='color:blue'>Approach 2: Use K Fold Cross validation</h3>

**Manually try suppling models with different parameters to cross_val_score function with 5 fold cross validation**

In [24]:
from sklearn.model_selection import cross_val_score

In [25]:
cross_val_score(svm.SVC(kernel="linear",C=10,gamma="auto"),iris.data,iris.target,cv=5)

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [26]:
cross_val_score(svm.SVC(kernel='rbf',C=10,gamma='auto'),iris.data, iris.target, cv=5)

array([0.96666667, 1.        , 0.96666667, 0.96666667, 1.        ])

In [27]:
cross_val_score(svm.SVC(kernel='rbf',C=20,gamma='auto'),iris.data, iris.target, cv=5)

array([0.96666667, 1.        , 0.9       , 0.96666667, 1.        ])

**Above approach is tiresome and very manual. We can use for loop as an alternative**

In [28]:
import numpy as np

In [29]:
kernels=["rbf","linear"]
C=[1,2,20]
avg_scores={}
for kval in kernels:
    for cval in C:
        cv_scores=cross_val_score(svm.SVC(kernel=kval,C=cval,gamma="auto"),iris.data,iris.target,cv=5)
        avg_scores[kval + "_" + str(cval)]=np.average(cv_scores)

avg_scores

{'rbf_1': np.float64(0.9800000000000001),
 'rbf_2': np.float64(0.9800000000000001),
 'rbf_20': np.float64(0.9666666666666668),
 'linear_1': np.float64(0.9800000000000001),
 'linear_2': np.float64(0.9800000000000001),
 'linear_20': np.float64(0.9666666666666666)}

**From above results we can say that rbf with C=1 or 10 or linear with C=1 will give best performance**

<h3 style='color:blue'>Approach 3: Use GridSearchCV</h3>

**GridSearchCV does exactly same thing as for loop above but in a single line of code**

In [31]:
dir(iris)

['DESCR',
 'data',
 'data_module',
 'feature_names',
 'filename',
 'frame',
 'target',
 'target_names']

In [32]:
from sklearn.model_selection import GridSearchCV
model=svm.SVC()
param_grid={
    'C':[1,10,20],
    'kernel':["linear","rbf"],
    "gamma":["auto"]
}
clf=GridSearchCV(model,param_grid=param_grid,cv=5,return_train_score=False)

clf.fit(iris.data,iris.target)


,estimator,SVC()
,param_grid,"{'C': [1, 10, ...], 'gamma': ['auto'], 'kernel': ['linear', 'rbf']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,1


In [33]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_tags__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_refit_for_multimetric',
 '_check_scorers_accept_sample_weight',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_params_html',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_html_repr',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_run_sea

In [34]:
df=pd.DataFrame(clf.cv_results_)
df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000768,0.000353,0.000617,0.000221,1,auto,linear,"{'C': 1, 'gamma': 'auto', 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.000623,0.000145,0.000405,0.000062,1,auto,rbf,"{'C': 1, 'gamma': 'auto', 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.000423,0.000030,0.000286,0.000016,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
3,0.000569,0.000173,0.000380,0.000089,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
4,0.000410,0.000053,0.000286,0.000059,20,auto,linear,"{'C': 20, 'gamma': 'auto', 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6


In [38]:
df[['param_C','param_kernel','param_gamma','mean_test_score']]

,param_C,param_kernel,param_gamma,mean_test_score
0,1,linear,auto,0.980000
1,1,rbf,auto,0.980000
2,10,linear,auto,0.973333
3,10,rbf,auto,0.980000
4,20,linear,auto,0.966667
5,20,rbf,auto,0.966667


In [36]:
clf.best_params_

{'C': 1, 'gamma': 'auto', 'kernel': 'linear'}

In [37]:
clf.best_score_

np.float64(0.9800000000000001)

In [ ]:
dir(clf)

**Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation**

In [39]:
from sklearn.model_selection import RandomizedSearchCV
model=svm.SVC()
param_grid={
    'C':[1,10,20],
    'kernel':["rbf","linear"],
    'gamma':["auto"]
}
rs=RandomizedSearchCV(model,param_distributions=param_grid,cv=5,return_train_score=False,n_iter=2)
rs.fit(iris.data,iris.target)
df=pd.DataFrame(rs.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kernel,param_gamma,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001817,0.002324,0.000497,0.000157,rbf,auto,20,"{'kernel': 'rbf', 'gamma': 'auto', 'C': 20}",0.966667,1.0,0.9,0.966667,1.0,0.966667,0.036515,1
1,0.000448,0.000049,0.000312,0.000033,linear,auto,20,"{'kernel': 'linear', 'gamma': 'auto', 'C': 20}",1.000000,1.0,0.9,0.933333,1.0,0.966667,0.042164,2


In [40]:
df[["param_C","param_kernel","param_gamma","mean_test_score"]]

,param_C,param_kernel,param_gamma,mean_test_score
0,20,rbf,auto,0.966667
1,20,linear,auto,0.966667


In [41]:
rs.best_params_

{'kernel': 'rbf', 'gamma': 'auto', 'C': 20}

In [42]:
rs.best_score_

np.float64(0.9666666666666668)

**How about different models with different hyperparameters?**

In [43]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


model_params={

    'SVM':{
        'model':svm.SVC(),
        'params':{
            'C':[1,10,20],
            'kernel':["rbf","linear"],
            'gamma':["auto"]
        }
    },
    'random_forest':{
        'model':RandomForestClassifier(),
        'params':{
            "n_estimators":[1,5,10]
        }
    },
    'logistic_regression':{
        'model':LogisticRegression(),
        'params':{
            "C":[1,5,10],
            "solver":["liblinear"],
            "multi_class":["auto"]

        }
    }
}

In [44]:
scores=[]

for model_name,mp in model_params.items():
    clf=GridSearchCV(estimator=mp['model'],param_grid=mp["params"],cv=5,return_train_score=False)
    clf.fit(iris.data,iris.target)
    scores.append({
        'model':model_name,
        'best_score':clf.best_score_,
        'best_params':clf.best_params_
    })


/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'

In [46]:
df=pd.DataFrame(scores,columns=["model","best_score","best_params"])
df

,model,best_score,best_params
0,SVM,0.980000,"{'C': 1, 'gamma': 'auto', 'kernel': 'rbf'}"
1,random_forest,0.966667,{'n_estimators': 10}
2,logistic_regression,0.966667,"{'C': 5, 'multi_class': 'auto', 'solver': 'lib..."


**Based on above, I can conclude that SVM with C=1 and kernel='rbf' is the best model for solving my problem of iris flower classification**